# Phát hiện đánh nhau trên video → chèn cảnh báo (Google Colab)

**Input**: 1 file video. **Output**: chính video đó (giữ nguyên âm thanh), ở các đoạn có đánh nhau thì hiện **banner đỏ "CẢNH BÁO: PHÁT HIỆN ĐÁNH NHAU"** và viền đỏ nhấp nháy quanh khung hình.

Model: **PaddleHuman ppTSM** (`models/ppTSM.onnx`), lấy từ `serving/model_engine/violence_paddlehuman/weights/` của repo gốc, chạy bằng `onnxruntime-gpu`. Không cần Docker hay Redis.

Cách chạy:
1. Đổi runtime sang GPU: **Runtime → Change runtime type → T4 GPU**.
2. Upload cả thư mục `fight_warning_video/` lên Google Drive (ví dụ `MyDrive/fight_warning_video`), hoặc nén thành `fight_warning_video.zip` rồi upload thẳng lên Colab.
3. Sửa đường dẫn ở **bước 2** và **bước 4**, rồi chạy lần lượt từ trên xuống.

Cấu trúc thư mục:
```
fight_warning_video/
├── run.py                  # CLI: python run.py --input in.mp4 --output out.mp4
├── models/ppTSM.onnx       # model
├── fight_warning/
│   ├── detector.py         # tiền xử lý + suy luận ONNX
│   ├── pipeline.py         # chấm điểm video, làm mượt, tìm đoạn đánh nhau, ghi video ra
│   └── overlay.py          # vẽ banner/viền cảnh báo
└── fight_warning_colab.ipynb
```

## 1. Kiểm tra GPU
Nếu báo lỗi `nvidia-smi: command not found` là bạn chưa bật GPU runtime. Vẫn chạy được bằng CPU nhưng rất chậm (xem mẹo ở cuối notebook).

In [ ]:
!nvidia-smi

## 2. Lấy code
**Cách A: Google Drive** (khuyên dùng, video output lưu lại trên Drive): để `USE_DRIVE = True` và sửa `PROJECT_DIR` cho khớp chỗ bạn upload thư mục.

**Cách B: upload file zip**: kéo `fight_warning_video.zip` vào thanh Files bên trái của Colab (thư mục `/content`), để `USE_DRIVE = False`. Cell sẽ tự giải nén.

In [ ]:
import os

USE_DRIVE = True
PROJECT_DIR = "/content/drive/MyDrive/fight_warning_video"   # <-- sửa cho khớp path thật trên Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    PROJECT_DIR = "/content/fight_warning_video"
    if not os.path.isdir(PROJECT_DIR) and os.path.isfile("/content/fight_warning_video.zip"):
        !unzip -q -o /content/fight_warning_video.zip -d /content

for f in ["run.py", "models/ppTSM.onnx", "fight_warning/pipeline.py"]:
    if not os.path.exists(os.path.join(PROJECT_DIR, f)):
        raise RuntimeError(f"Không thấy {f} trong {PROJECT_DIR}. Kiểm tra lại PROJECT_DIR "
                           "(vd chạy !ls /content/drive/MyDrive để xem tên thư mục thật).")

%cd $PROJECT_DIR
!ls -la

## 3. Cài thư viện
Colab đã có sẵn `opencv`, `numpy`, `pillow`, `matplotlib`, `ffmpeg`, nên chỉ cần cài thêm `onnxruntime-gpu`. Dòng cuối phải in ra có `CUDAExecutionProvider` thì mới chạy được GPU.

In [ ]:
!pip uninstall -y -q onnxruntime 2>/dev/null
!pip install -q onnxruntime-gpu

import onnxruntime as ort
print("onnxruntime", ort.__version__, ort.get_available_providers())

## 4. Chọn video và tham số
- `INPUT_VIDEO`: đường dẫn video đầu vào (trên Drive hoặc `/content/...`). Đặt `UPLOAD_FROM_PC = True` nếu muốn chọn file từ máy tính.
- `OUTPUT_VIDEO`: để `None` thì tự lưu vào `output/<tên video>_warning.mp4` trong `PROJECT_DIR`.
- Tham số mặc định giống `violence-paddlehuman-business-service` của repo gốc (threshold 0.65, làm mượt 3 cửa sổ).

In [ ]:
import os

INPUT_VIDEO = "/content/drive/MyDrive/videos/demo1.mp4"   # <-- sửa path video đầu vào
UPLOAD_FROM_PC = False                                     # True: chọn file video từ máy tính
OUTPUT_VIDEO = None                                        # None = output/<tên>_warning.mp4

THRESHOLD = 0.65         # điểm fight >= ngưỡng thì coi là đánh nhau (tăng lên nếu báo nhầm nhiều)
SMOOTHING_WINDOW = 3     # làm mượt điểm trên N lần chấm liên tiếp
MIN_EVENT_SEC = 1.0      # bỏ các đoạn đánh nhau ngắn hơn N giây (lọc báo động giả)
MERGE_GAP_SEC = 1.0      # gộp 2 đoạn đánh nhau nếu cách nhau ít hơn N giây
STRIDE = 8               # cứ mỗi N frame chấm điểm 1 lần (tính ở 25fps; tăng lên nếu chạy CPU)
DEVICE = "auto"          # auto | cuda | cpu

WARNING_TEXT = "CẢNH BÁO: PHÁT HIỆN ĐÁNH NHAU"
SHOW_SCORE = True        # hiện điểm fight trên banner cảnh báo
SHOW_HUD = False         # hiện điểm fight nhỏ ở góc cả khi KHÔNG đánh nhau (để debug)
KEEP_AUDIO = True        # giữ âm thanh gốc

if UPLOAD_FROM_PC:
    import shutil
    from google.colab import files
    os.makedirs("/content/input", exist_ok=True)
    uploaded = files.upload()          # file được lưu vào thư mục hiện tại (PROJECT_DIR)
    name = next(iter(uploaded))
    INPUT_VIDEO = os.path.join("/content/input", name)
    shutil.move(name, INPUT_VIDEO)

if not os.path.isfile(INPUT_VIDEO):
    raise FileNotFoundError(f"Không thấy video: {INPUT_VIDEO}")
if OUTPUT_VIDEO is None:
    OUTPUT_VIDEO = os.path.join(PROJECT_DIR, "output",
                                os.path.splitext(os.path.basename(INPUT_VIDEO))[0] + "_warning.mp4")
REPORT_JSON = os.path.splitext(OUTPUT_VIDEO)[0] + "_report.json"
print("Input :", INPUT_VIDEO)
print("Output:", OUTPUT_VIDEO)

## 5. Chạy
Chạy 2 lượt: **[1/2]** đọc video và chấm điểm fight theo từng cửa sổ ~2 giây, **[2/2]** đọc lại video và vẽ cảnh báo lên các đoạn đánh nhau. Kết thúc sẽ in danh sách các đoạn phát hiện được.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-u", "run.py",
    "--input", INPUT_VIDEO,
    "--output", OUTPUT_VIDEO,
    "--report", REPORT_JSON,
    "--threshold", str(THRESHOLD),
    "--smoothing-window", str(SMOOTHING_WINDOW),
    "--min-event-sec", str(MIN_EVENT_SEC),
    "--merge-gap-sec", str(MERGE_GAP_SEC),
    "--stride", str(STRIDE),
    "--device", DEVICE,
    "--warning-text", WARNING_TEXT,
    "--show-score", str(SHOW_SCORE),
    "--show-hud", str(SHOW_HUD),
    "--keep-audio", str(KEEP_AUDIO),
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(f"run.py lỗi (exit code {proc.returncode}), xem log ở trên.")

## 6. Xem kết quả: các đoạn đánh nhau và biểu đồ điểm
Đường đỏ nét đứt là ngưỡng, vùng tô đỏ là các đoạn được gắn cảnh báo. Dùng biểu đồ này để chỉnh `THRESHOLD` / `MIN_EVENT_SEC` ở bước 4 rồi chạy lại bước 5.

In [ ]:
import json
import matplotlib.pyplot as plt

with open(REPORT_JSON, encoding="utf-8") as f:
    report = json.load(f)

print(f"Video: {report['duration_sec']}s, {report['n_frames']} frame @ {report['fps']:.2f} fps, "
      f"{report['resolution'][0]}x{report['resolution'][1]} | providers: {report['providers']}")
print(f"Xử lý mất {report['elapsed_sec']}s\n")
segs = report["fight_segments"]
if segs:
    print(f"{'#':>3} {'bắt đầu':>9} {'kết thúc':>9} {'dài':>7} {'max':>6} {'tb':>6}")
    for i, s in enumerate(segs, 1):
        print(f"{i:>3} {s['start_sec']:>8.2f}s {s['end_sec']:>8.2f}s {s['duration_sec']:>6.2f}s "
              f"{s['max_score']:>6.2f} {s['mean_score']:>6.2f}")
else:
    print("Không phát hiện đoạn đánh nhau nào.")

t = [w["time_sec"] for w in report["windows"]]
plt.figure(figsize=(14, 3.5))
plt.plot(t, [w["score"] for w in report["windows"]], color="#999999", lw=1, label="điểm thô")
plt.plot(t, [w["smoothed"] for w in report["windows"]], color="#1f77b4", lw=2, label="điểm đã làm mượt")
plt.axhline(report["settings"]["threshold"], color="red", ls="--", lw=1, label="ngưỡng")
for s in segs:
    plt.axvspan(s["start_sec"], s["end_sec"], color="red", alpha=0.15)
plt.ylim(-0.02, 1.02); plt.xlim(0, report["duration_sec"])
plt.xlabel("thời gian (giây)"); plt.ylabel("fight score"); plt.legend(loc="upper right"); plt.grid(alpha=0.3)
plt.show()

## 7. Xem nhanh video output
Hiện 1 khung hình trong mỗi đoạn đánh nhau (tối đa 4), và phát video ngay trong notebook nếu file nhỏ hơn 50 MB (file lớn hơn thì tải về ở bước 8 để xem).

In [ ]:
import base64, os
import cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display

cap = cv2.VideoCapture(OUTPUT_VIDEO)
for s in report["fight_segments"][:4]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, (s["start_frame"] + s["end_frame"]) // 2)
    ok, frame = cap.read()
    if ok:
        plt.figure(figsize=(10, 6)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis("off")
        plt.title(f"{s['start_sec']:.1f}s - {s['end_sec']:.1f}s"); plt.show()
cap.release()

size_mb = os.path.getsize(OUTPUT_VIDEO) / 1e6
if size_mb <= 50:
    b64 = base64.b64encode(open(OUTPUT_VIDEO, "rb").read()).decode()
    display(HTML(f'<video width="800" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print(f"Video {size_mb:.0f} MB, quá lớn để phát trong notebook. Tải về ở bước 8.")

## 8. Tải video output về máy
Nếu dùng Drive thì file đã nằm sẵn trong `PROJECT_DIR/output/` trên Drive, không cần tải.

In [ ]:
from google.colab import files
files.download(OUTPUT_VIDEO)
files.download(REPORT_JSON)

## (Tuỳ chọn) Xử lý cả thư mục video
Chạy lần lượt mọi video trong `INPUT_DIR` với cùng tham số ở bước 4, output lưu vào `PROJECT_DIR/output/`.

In [ ]:
import glob, os, subprocess, sys

INPUT_DIR = "/content/drive/MyDrive/videos"   # <-- thư mục chứa video
videos = sorted(p for ext in ("mp4", "avi", "mov", "mkv") for p in glob.glob(os.path.join(INPUT_DIR, f"*.{ext}")))
print(f"{len(videos)} video")
for v in videos:
    out = os.path.join(PROJECT_DIR, "output", os.path.splitext(os.path.basename(v))[0] + "_warning.mp4")
    print(f"\n===== {v}")
    subprocess.run([sys.executable, "-u", "run.py", "--input", v, "--output", out,
                    "--threshold", str(THRESHOLD), "--smoothing-window", str(SMOOTHING_WINDOW),
                    "--min-event-sec", str(MIN_EVENT_SEC), "--merge-gap-sec", str(MERGE_GAP_SEC),
                    "--stride", str(STRIDE), "--device", DEVICE, "--warning-text", WARNING_TEXT,
                    "--show-score", str(SHOW_SCORE), "--show-hud", str(SHOW_HUD),
                    "--keep-audio", str(KEEP_AUDIO)], check=False)

## Mẹo và lưu ý
- **Báo nhầm nhiều**: tăng `THRESHOLD` (vd 0.75–0.85) hoặc tăng `MIN_EVENT_SEC`. **Bỏ sót**: giảm `THRESHOLD` (vd 0.5).
- **Cảnh báo bị đứt quãng** trong 1 trận đánh: tăng `MERGE_GAP_SEC` (vd 2.0).
- **Chạy CPU** (không có GPU): mỗi lần chấm điểm mất vài giây, nên đặt `STRIDE = 25` (chấm 1 lần/giây) để đỡ chậm.
- Model chỉ nhìn **vùng giữa khung hình**: giống serving gốc, frame bị resize về 341x256 rồi cắt giữa 224x224, nên với video 16:9 khoảng 1/6 mỗi mép trái/phải nằm ngoài vùng model nhìn thấy. Đánh nhau sát mép hình dễ bị bỏ sót.
- `sample_freq`/`stride` được tính ở 25 fps và tự co giãn theo fps thật của video, nên mỗi clip luôn phủ khoảng 2.2 giây dù video 25, 30 hay 60 fps.
- Chạy từ terminal (không cần notebook): `python run.py --input in.mp4 --output out.mp4` (thêm `--help` để xem hết tham số).